# Lekce 01 - Úvod do AI agentů

Vítejte v první lekci kurzu **AI agenti pro začátečníky**!

**AI agent** je program, který používá velký jazykový model (LLM) jako svůj logický motor a může v reálném světě provádět *akcí* — volat API, dotazovat se v databázích nebo spouštět kód — za cílem dosažení úkolu pro uživatele.

V tomto poznámkovém bloku si vytvoříte svého prvního agenta: **cestovního agenta**, který doporučuje prázdninové destinace. Během toho se naučíte:

1. Připojit se k Azure AI Foundry Agent Service pomocí **Microsoft Agent Framework**.
2. Poskytnout agentovi **nástroj** — obyčejnou Python funkci, kterou může volat.
3. Spustit agenta a zkontrolovat jeho odpověď.
4. Proudit odpověď agenta token po tokenu.


## Nastavení

Než spustíte tento notebook, ujistěte se, že máte:

1. **Projekt Azure AI Foundry** s nasazeným modelem chatu (např. `gpt-4o-mini`).
2. **Přihlášení pomocí Azure CLI** — spusťte `az login` ve vašem terminálu.
3. **Nastavené požadované proměnné prostředí:**
   - `AZURE_AI_PROJECT_ENDPOINT` — endpoint vašeho projektu Azure AI Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — název nasazeného modelu.

Buňka níže nainstaluje potřebné Python balíčky.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Vytvoření vašeho prvního agenta

Agent potřebuje dvě věci:

- **Instrukce**, které mu říkají, *kdo je* a *jak se má chovat* (systémový prompt).
- **Nástroje** — Python funkce označené dekorátorem `@tool`, které může agent volat, aby získal informace nebo provedl akce.

Níže definujeme jednoduchý nástroj, který vrací seznam oblíbených dovolenkových destinací. Agent tento nástroj použije, když uživatel požádá o doporučení na cestování.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming Responses

Pro interaktivnější zážitek můžete **streamovat** odpověď agenta. Místo čekání na celou odpověď agent postupně odesílá textové úryvky, jak jsou generovány. To je zvláště užitečné v chatovacích rozhraních, kde chcete zobrazit výstup v reálném čase.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Shrnutí

V této lekci jste se naučili, jak:

- **Vytvořit poskytovatele** který se připojuje k Azure AI Foundry Agent Service pomocí `AzureAIProjectAgentProvider`.
- **Definovat nástroj** pomocí dekorátoru `@tool`, aby agent mohl volat vaše Python funkce.
- **Spustit agenta** se zprávou uživatele a vytisknout jeho odpověď.
- **Streamovat odpovědi** pro výstup v reálném čase.

V další lekci prozkoumáme agentní rámce podrobněji a naučíme se, jak agentům poskytnout výkonnější nástroje a schopnosti vícekrokového uvažování.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Prohlášení o vyloučení odpovědnosti**:  
Tento dokument byl přeložen pomocí AI překladatelské služby [Co-op Translator](https://github.com/Azure/co-op-translator). Přestože usilujeme o přesnost, mějte prosím na paměti, že automatické překlady mohou obsahovat chyby nebo nepřesnosti. Původní dokument v jeho mateřském jazyce by měl být považován za závazný zdroj. Pro kritické informace je doporučen profesionální lidský překlad. Nejsme odpovědni za žádné nedorozumění nebo mylné výklady vyplývající z použití tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
